# Inference
Esta sección se evalúa usando la métrica de Accuracy. Se puede ver como una prueba de clasificación sobre 3 clases.

In [ ]:
import pandas as pd
import re
from sklearn.metrics import classification_report, confusion_matrix

def get_infer(path, test):
    if test:
        start = 226 
        mid = 452
    else:
        start = 203
        mid = 406
    dataset = pd.read_csv(path)
    dataset = dataset.drop(columns = ["Unnamed: 0"])
    print("Longitud del dataset: {}".format(len(dataset)))
    infer = dataset[start:mid]
    infer = infer.reset_index()
    infer = infer.drop(columns=["index"])
    return infer

llm_test_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv", True)
llm_dev_ds = get_infer("/home/flopezp/Kurosagol/Ongoing/baseline_datasets/validation/raw/DeepSeek-R1-0528-Qwen3-8B_full.csv", False)

folio_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
test_values = [1 if folio_test["label"].iloc[i] == 'True' else 0 if folio_test["label"].iloc[i] == 'False' else 2 for i in range(len(folio_test))]

folio_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
val_values = [1 if folio_val["label"].iloc[i] == 'True' else 0 if folio_val["label"].iloc[i] == 'False' else 2 for i in range(len(folio_val))]

Longitud del dataset: 678
Longitud del dataset: 609


In [25]:
for i in range(len(llm_test_ds["Full"])):
    print(i)
    print(llm_test_ds["Full"][i])
    print('========================================')

0
    Let's think step by step:
    From the premises, we know that there exists an x such that Athlete(x) and Professional(x) and CenterBack(x). This doesn't directly say anything about Stephen Curry.
    We also know that Stephen Curry plays professional basketball. But the premises about soccer players and defenders are about soccer, not basketball.
    There is no premise that connects basketball to the other attributes. Therefore, we cannot conclude anything about Stephen Curry's attributes from the premises.
    However, the conclusion is about Athlete(stephenCurry). We don't have information to confirm or deny this. But note that the premises do not contradict it. 
    For example, if Stephen Curry were an athlete, it would be consistent, but we don't know. Similarly, if he were not, it would also be consistent.
    Therefore, the conclusion is Uncertain.

    But wait, let's look at the premises again. The premises are about soccer players and defenders, but Stephen Curry is pl

In [50]:
pred_dict = {
    'True': 1,
    'False': 0,
    'Uncertain': 2
}

keys = list(pred_dict.keys())

predictions = []
for i in range(len(llm_test_ds["Full"])):
    aux = llm_test_ds["Full"][i].split('\n')
    aux2 = [re.sub('  ', '', x) for x in aux]
    aux3 = [x for x in aux2 if x != '']
    counter = False
    for elem in keys:
        if aux3[0] == elem:
            predictions.append(pred_dict[elem])
            counter = True
    if not counter:
        predictions.append(-1)
    
    if not counter:
        split1 = llm_test_ds["Full"][i].split('-----')[0]
        split2 = split1.split('\n')
        split3 = [re.sub('  ', '', x) for x in split2]
        split4 = [x for x in split3 if x != '']
        tru_fal_unc = [0,0,0]
        for value in split4[-10:]:
            if 'True' in value or 'true' in value:
                tru_fal_unc[1] += 1
            elif 'Uncertain' in value or 'uncertain' in value:
                tru_fal_unc[2] += 1
            elif 'False' in value or 'false' in value:
                tru_fal_unc[0] += 1
        #print("True {}. False {}. Uncertain {}.".format(tru_fal_unc[1], tru_fal_unc[2], tru_fal_unc[0]))
        second_counter = False
        for elem in keys:
            if elem in split4[-1]:
                predictions[i] = pred_dict[elem]
                second_counter = True
        if not second_counter:
            max_value = tru_fal_unc.index(max(tru_fal_unc))
            predictions[i] = max_value

            
        #print(split4[-1])
        #print(split4[-10:])
        #print('-----')

#    print(aux3)
#    print('-----------------------')
print(len(predictions) - predictions.count(-1))
print(predictions.count(-1))


226
0


In [ ]:
# Hmmmmmmm sussy baky, pero vamos bien.
def evaluate_acc(lista):
    print(classification_report(lista, test_values))
    print(confusion_matrix(lista, test_values))

evaluate_acc(predictions)

              precision    recall  f1-score   support

           0       0.54      0.58      0.56        57
           1       0.80      0.56      0.66       125
           2       0.36      0.64      0.46        44

    accuracy                           0.58       226
   macro avg       0.57      0.59      0.56       226
weighted avg       0.65      0.58      0.60       226

[[33 11 13]
 [18 70 37]
 [10  6 28]]


In [4]:
def get_labels(dataset, test):
    values = []
    for i in range(len(dataset)):
        text = dataset["Full"].iloc[i]
        text = re.sub('\n', '', text)
        unc = (text.count('Uncertain') + text.count('uncertain'), 'Uncertain')
        false = (text.count('False') + text.count('false'), 'False')
        true = (text.count('True') + text.count('true'), 'True')
        biggest = max(unc, false, true, key=lambda x:x[0])
        #print(f"Label: {biggest[1]}. T = {true[0]}, F= {false[0]}, U = {unc[0]}")
        if biggest[1] == 'True':
            values.append(1)
        elif biggest[1] == 'False':
            values.append(0)
        else:
            values.append(2)

    if test:
        print("------------------")
        print("====== TEST ======")
        print("------------------")
        print(classification_report(values, test_values))
        print(confusion_matrix(values, test_values))
    else:
        print("-----------------")
        print("====== VAL ======")
        print("-----------------")
        print(classification_report(values, val_values))
        print(confusion_matrix(values, val_values))

get_labels(llm_test_ds, True)
get_labels(llm_dev_ds, False)

------------------
====== TEST ======
------------------
              precision    recall  f1-score   support

           0       0.34      0.46      0.39        46
           1       0.87      0.48      0.62       158
           2       0.19      0.68      0.30        22

    accuracy                           0.50       226
   macro avg       0.47      0.54      0.44       226
weighted avg       0.70      0.50      0.54       226

[[21  8 17]
 [36 76 46]
 [ 4  3 15]]
-----------------
====== VAL ======
-----------------
              precision    recall  f1-score   support

           0       0.47      0.59      0.52        49
           1       0.83      0.48      0.61       126
           2       0.28      0.68      0.39        28

    accuracy                           0.53       203
   macro avg       0.53      0.58      0.51       203
weighted avg       0.67      0.53      0.56       203

[[29  7 13]
 [29 60 37]
 [ 4  5 19]]
